# Week 1 Capstone — 5 Stock Metrics Summary Table
**Aayan Mulani │ Decimal Point Analytics Preparation**

Pulling together everything from Week 1 — returns, volatility,
Sharpe Ratio, Beta, and Win Rate — into one clean professional
metrics table across 5 Indian stocks (2020–2025).

In [6]:
import pandas as pd
import numpy as np
import yfinance as yf

tickers = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ITC.NS']

# Download 5 years of data
stocks = yf.download(tickers, start='2020-01-01', end='2025-01-01')['Close']
nifty  = yf.download('^NSEI',  start='2020-01-01', end='2025-01-01')['Close'].squeeze()

print(stocks.shape)
print(nifty.shape)

[*********************100%***********************]  5 of 5 completed
[*********************100%***********************]  1 of 1 completed

(1238, 5)
(1237,)


In [10]:
# Align stocks and nifty on common dates only
stocks, nifty = stocks.align(nifty, join='inner', axis=0)

# Compute daily returns
stock_returns = stocks.pct_change().dropna()
nifty_returns = nifty.pct_change().dropna()

print(stock_returns.shape)
print(nifty_returns.shape)
print(stock_returns.head())

(1236, 5)
(1236,)
Ticker      HDFCBANK.NS   INFY.NS    ITC.NS  RELIANCE.NS    TCS.NS
Date                                                              
2020-01-02     0.006374 -0.002918  0.007350     0.017025 -0.004590
2020-01-03    -0.014261  0.015381 -0.005628     0.001205  0.019929
2020-01-06    -0.021642 -0.009584 -0.014256    -0.023192 -0.000091
2020-01-07     0.015835 -0.014820  0.001063     0.015385  0.002454
2020-01-08    -0.002618 -0.013326 -0.004886    -0.007510  0.022395


## Computing Metrics
Building annualised return, volatility, Sharpe Ratio, Beta,
and Win Rate for each stock across 5 years (2020–2025).

In [13]:
trading_days = 252

annualised_return = stock_returns.mean() * trading_days
annualised_vol    = stock_returns.std() * np.sqrt(trading_days)
sharpe_ratio      = annualised_return / annualised_vol

print("Annualised Return:\n", annualised_return.round(4))
print("\nAnnualised Volatility:\n", annualised_vol.round(4))
print("\nSharpe Ratio:\n", sharpe_ratio.round(4))

Annualised Return:
 Ticker
HDFCBANK.NS    0.1123
INFY.NS        0.2548
ITC.NS         0.2240
RELIANCE.NS    0.1631
TCS.NS         0.1796
dtype: float64

Annualised Volatility:
 Ticker
HDFCBANK.NS    0.2729
INFY.NS        0.2786
ITC.NS         0.2588
RELIANCE.NS    0.2974
TCS.NS         0.2448
dtype: float64

Sharpe Ratio:
 Ticker
HDFCBANK.NS    0.4115
INFY.NS        0.9146
ITC.NS         0.8657
RELIANCE.NS    0.5484
TCS.NS         0.7335
dtype: float64


In [14]:
# Beta = Covariance(stock, nifty) / Variance(nifty)
nifty_var = nifty_returns.var()

betas = {}
for col in stock_returns.columns:
    cov = stock_returns[col].cov(nifty_returns)
    betas[col] = cov / nifty_var

beta_series = pd.Series(betas)
print("Beta:\n", beta_series.round(4))

Beta:
 HDFCBANK.NS    1.0858
INFY.NS        0.8931
ITC.NS         0.6779
RELIANCE.NS    1.1125
TCS.NS         0.7511
dtype: float64


In [15]:
# Win Rate = percentage of days with positive return
win_rate = (stock_returns > 0).mean()
print("Win Rate:\n", win_rate.round(4))

Win Rate:
 Ticker
HDFCBANK.NS    0.5194
INFY.NS        0.5235
ITC.NS         0.5057
RELIANCE.NS    0.5202
TCS.NS         0.5170
dtype: float64


## Final Metrics Summary Table
Consolidating all 5 metrics into one DataFrame for clean comparison.

In [18]:
summary = pd.DataFrame({
    'Annualised Return %' : (annualised_return * 100).round(2),
    'Annualised Vol %'    : (annualised_vol * 100).round(2),
    'Sharpe Ratio'        : sharpe_ratio.round(2),
    'Beta'                : beta_series.round(2),
    'Win Rate %'          : (win_rate * 100).round(2)
})

print(summary.to_string())

             Annualised Return %  Annualised Vol %  Sharpe Ratio  Beta  Win Rate %
HDFCBANK.NS                11.23             27.29          0.41  1.09       51.94
INFY.NS                    25.48             27.86          0.91  0.89       52.35
ITC.NS                     22.40             25.88          0.87  0.68       50.57
RELIANCE.NS                16.31             29.74          0.55  1.11       52.02
TCS.NS                     17.96             24.48          0.73  0.75       51.70


## Summary — Week 1 Capstone

Five year analysis (2020–2025) across RELIANCE, TCS, INFY, HDFCBANK, ITC.

Key findings:
- INFY: Best risk-adjusted performer (Sharpe 0.91) despite weak 3-year chart
- ITC: Most defensive stock (Beta 0.68, Vol 25.88%) — ideal for low-risk portfolios  
- HDFCBANK: Worst overall — high beta, low return, lowest Sharpe (0.41)
- Win rates cluster tightly 50.5–52.4% across all stocks — consistent with EMH
- Timeframe matters — 3yr vs 5yr analysis tells completely different stories

Metrics used: Annualised Return, Annualised Volatility, Sharpe Ratio, Beta, Win Rate